# DNN to perform MPPI

In [29]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt

### Data Preprocessing

In [41]:
x_csv_path = 'data/inputs.csv'
y_csv_path = 'data/outputs.csv'
# Import CSV file into a DataFrame
x_df = pd.read_csv(x_csv_path)
y_df = pd.read_csv(y_csv_path)
x = x_df.to_numpy()
y = y_df.to_numpy()
print('shape of x is : ',x.shape)
print('shape of y is : ',y.shape)

shape of x is :  (897, 261)
shape of y is :  (897, 20)


In [42]:
class diabetesdataset(Dataset):
  def __init__(self,x,y):
    self.x = torch.tensor(x,dtype=torch.float32)
    self.y = torch.tensor(y,dtype=torch.float32)
    self.length = self.x.shape[0]
  def __getitem__(self,idx):
    return self.x[idx],self.y[idx]
  def __len__(self):
    return self.length
dataset = diabetesdataset(x,y)

# Split dataset into train and test sets
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create DataLoader for train and test sets
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [52]:
# Network Architecture
class mppinet(nn.Module):
    def __init__(self):
        super(mppinet, self).__init__()
        self.fc1 = nn.Linear(261, 512)  # Assuming input size is 261
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 20)  # Output layer with 20 units for action pairs

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        x = self.fc5(x)
        return x

# Instantiate the model
model = mppinet()

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Training Network

In [53]:
# Function to calculate accuracy
def calculate_accuracy(outputs, labels, threshold=0.1):
    correct = ((outputs - labels).abs() < threshold).all(dim=1)
    return correct.float().mean().item()

In [55]:
# Training loop
num_epochs = 1000
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []
for epoch in range(num_epochs):
    # Training
    model.train()
    running_train_loss = 0.0
    running_train_accuracy = 0.0
    for inputs, labels in train_dataloader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * inputs.size(0)
        accuracy = calculate_accuracy(outputs, labels)
        running_train_accuracy += accuracy * inputs.size(0)
    epoch_train_loss = running_train_loss / len(train_dataset)
    epoch_train_accuracy = running_train_accuracy / len(train_dataset)
    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_accuracy)

    # Testing
    model.eval()
    running_test_loss = 0.0
    running_test_accuracy = 0.0
    with torch.no_grad():
        for inputs, labels in test_dataloader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_test_loss += loss.item() * inputs.size(0)
            accuracy = calculate_accuracy(outputs, labels)
            running_test_accuracy += accuracy * inputs.size(0)
    epoch_test_loss = running_test_loss / len(test_dataset)
    epoch_test_accuracy = running_test_accuracy / len(test_dataset)
    test_losses.append(epoch_test_loss)
    test_accuracies.append(epoch_test_accuracy)

    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {epoch_train_loss:.4f}, '
          f'Test Loss: {epoch_test_loss:.4f}, '
          f'Train Accuracy: {epoch_train_accuracy:.4f}, '
          f'Test Accuracy: {epoch_test_accuracy:.4f}')

# Plot the losses
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Test Losses')
plt.legend()
plt.show()

# Plot the accuracies
plt.figure(figsize=(10, 5))
plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(test_accuracies, label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Test Accuracies')
plt.legend()
plt.show()

Epoch [1/1000], Train Loss: 0.0045, Test Loss: 0.0374, Train Accuracy: 0.4533, Test Accuracy: 0.3389
Epoch [2/1000], Train Loss: 0.0045, Test Loss: 0.0363, Train Accuracy: 0.4630, Test Accuracy: 0.3167
Epoch [3/1000], Train Loss: 0.0052, Test Loss: 0.0375, Train Accuracy: 0.4254, Test Accuracy: 0.3667
Epoch [4/1000], Train Loss: 0.0053, Test Loss: 0.0358, Train Accuracy: 0.4142, Test Accuracy: 0.3333
Epoch [5/1000], Train Loss: 0.0047, Test Loss: 0.0357, Train Accuracy: 0.4742, Test Accuracy: 0.3500
Epoch [6/1000], Train Loss: 0.0040, Test Loss: 0.0358, Train Accuracy: 0.4854, Test Accuracy: 0.3500
Epoch [7/1000], Train Loss: 0.0043, Test Loss: 0.0356, Train Accuracy: 0.4603, Test Accuracy: 0.3611
Epoch [8/1000], Train Loss: 0.0038, Test Loss: 0.0364, Train Accuracy: 0.4937, Test Accuracy: 0.3056
Epoch [9/1000], Train Loss: 0.0036, Test Loss: 0.0366, Train Accuracy: 0.5230, Test Accuracy: 0.3500
Epoch [10/1000], Train Loss: 0.0042, Test Loss: 0.0365, Train Accuracy: 0.4854, Test Accura

KeyboardInterrupt: 

In [51]:
with torch.no_grad():
        for inputs, labels in test_dataloader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_test_loss += loss.item() * inputs.size(0)
            accuracy = calculate_accuracy(outputs, labels)
            running_test_accuracy += accuracy * inputs.size(0)
            print(outputs)

tensor([[ 1.0045,  0.1773,  1.0012, -0.6711,  0.9985,  0.7146,  1.0005, -0.0933,
          1.0027, -2.1186,  0.9981, -0.5858,  1.0026,  0.8256,  1.0028,  0.0574,
          1.0044, -0.9058,  0.9990, -0.3992],
        [ 1.0036,  0.5647,  1.0004, -0.2219,  1.0001,  0.0910,  1.0025,  0.0308,
          1.0030, -2.2151,  1.0023, -0.5668,  1.0024,  1.2935,  1.0035,  0.4739,
          1.0041, -0.5878,  0.9998,  0.1368],
        [ 1.0036,  0.0644,  0.9949, -0.1462,  0.9982,  0.2250,  1.0025,  0.0670,
          1.0034, -2.3507,  1.0015, -0.4762,  1.0033,  1.3523,  0.9998,  0.7217,
          1.0061, -1.0791,  0.9978, -0.6996],
        [ 1.0004,  0.4151,  1.0009, -0.5827,  0.9968,  0.8967,  0.9970,  0.0212,
          0.9996, -2.1702,  0.9946, -0.5950,  0.9991,  0.9416,  1.0011, -0.0074,
          1.0018, -0.9282,  0.9969, -0.3104],
        [ 1.0011, -0.1792,  1.0013, -0.2168,  1.0013,  0.2050,  1.0006, -0.3206,
          1.0001, -0.4167,  0.9989, -0.1349,  1.0021, -0.0371,  1.0019, -0.1514,
      